In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Set paths
BASE_DIR = '/content/drive/MyDrive/fake_news_data'
data_path = os.path.join(BASE_DIR, 'combined_fake_news_preprocessed.csv')
models_dir = os.path.join(BASE_DIR, 'models')
os.makedirs(models_dir, exist_ok=True)

In [ ]:
# Load and split the dataset
df = pd.read_csv(data_path)
df = df[['cleaned_text', 'label']].copy()
df = df[df['cleaned_text'].str.strip() != '']

X = df['cleaned_text'].values
y = df['label'].values

# Use the same split as before
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# Tokenize the texts (DistilBERT tokenizer)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
max_length = 512

def tokenize_function(texts):
    return tokenizer(
        texts.tolist(),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

train_encodings = tokenize_function(X_train)
val_encodings = tokenize_function(X_val)
test_encodings = tokenize_function(X_test)

# Create PyTorch datasets
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FakeNewsDataset(train_encodings, y_train)
val_dataset = FakeNewsDataset(val_encodings, y_val)
test_dataset = FakeNewsDataset(test_encodings, y_test)

In [ ]:
# Load DistilBERT model for sequence classification
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

In [ ]:
# Define metrics function for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

In [ ]:
# Set up training arguments and Trainer
training_args = TrainingArguments(
    output_dir=os.path.join(models_dir, 'distilbert_checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    save_total_limit=2,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# Train model
trainer.train()

In [ ]:
# Evaluate on the test set
test_results = trainer.evaluate(test_dataset)
print("\nTest set evaluation:")
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"F1-score:  {test_results['eval_f1']:.4f}")

In [ ]:
# Save the fine-tuned model and tokenizer to Drive
model.save_pretrained(os.path.join(models_dir, 'distilbert_fake_news'))
tokenizer.save_pretrained(os.path.join(models_dir, 'distilbert_fake_news'))

print("Model and tokenizer saved to", os.path.join(models_dir, 'distilbert_fake_news'))